# Notebook 20: qHTS auxiliary heads that can see inactive compounds

Every multitask network in this project (nb5, nb6, nb10A, nb19) builds its five AID 1851
heads from `Fit_LogAC50` and drops every row where no curve fitted. Counted on the cached
table, that keeps 9,841 / 8,317 / **5,974** / 9,381 of 17,121 rows for
1A2 / 2C9 / **2D6** / 3A4. The auxiliary heads therefore learn potency among actives only,
which is the same curve-fit selection bias the challenge labels carry, and it is worst on
CYP2D6, our weakest endpoint. `Max_Response` exists for all 17,121 rows. The rank-19 entry
uses it.

This is **notebook 5 with one change**: the qHTS target table. Warm start
(`chemprop_medium`), architecture, the four challenge and four screen heads, folds, epochs,
seed and checkpointing are copied from nb5, the run that produced `mtl13`.

Set `VARIANT` in the setup cell:

| `VARIANT` | qHTS heads | arm name |
| :--- | :--- | :--- |
| `'mr'` (default) | 5 `Max_Response` heads on every row | `mtlmr` |
| `'both'` | 5 `Fit_LogAC50` heads plus 5 `Max_Response` heads | `mtlboth` |
| `'ac50'` | nb5's table exactly, the control | `mtlac50` |

**Reference to beat**, `mtl13`: `0.5577 / 0.6935 / 0.4515 / 0.8022`, macro dev
**0.6262**, macro holdout **0.6652**. Gate: macro dev at least +0.015 with CYP2D6 up;
anything smaller is inside seed noise.

**Safe to run beside notebook 19**, in a second Colab runtime. It reads nothing nb19
writes, and every file it writes carries its own name: `nb20_<VARIANT>_state.npz`,
`<arm>_<iso>.npz`, `results_nb20_<VARIANT>.json`, and its own AID 1851 cache
`aid1851_nb20.csv`, which is written to a temporary name and renamed so a half-written
file is never read. The three variants can also run beside each other.

**Runtime** about 3 hours for `mr` (roughly 15% more public compounds than nb5).
Checkpointed per fold to Drive; Run all resumes.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
VARIANT = 'mr'          # 'mr', 'both' or 'ac50'; see the table above
ARM = {'mr': 'mtlmr', 'both': 'mtlboth', 'ac50': 'mtlac50'}[VARIANT]
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
print('variant', VARIANT, '| arm', ARM, '| writing to', OUTDIR)

NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy',
        'scaffold_groups.npy', 'aux_train.npy']
have = sorted(os.listdir('/content/cache'))
missing = [f for f in NEED if f not in have]
assert not missing, 'payload is missing %s. Found: %s' % (missing, have)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK='/content/chemprop_medium.pt'
if not Path(CK).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/checkpoints/chemprop_medium.pt', CK)
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB')

# Own cache name, written atomically: nb19 may be downloading its own copy into the same
# Drive folder at the same moment, and neither notebook should ever read the other's.
URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
AID = OUT / 'aid1851_nb20.csv'
def _aid_ok(p):
    try:
        cols = pd.read_csv(p, nrows=0).columns
        return {'Fit_LogAC50', 'Max_Response', 'Panel Name'} <= set(cols)
    except Exception:
        return False
if not (AID.exists() and _aid_ok(AID)):
    print('downloading AID 1851 (first run only; cached on Drive after this)')
    tmp = Path('/content/aid1851_dl.csv')
    urllib.request.urlretrieve(URL, str(tmp))
    assert _aid_ok(tmp), 'AID 1851 download lacks Fit_LogAC50 / Max_Response'
    import uuid; tmp_drive = OUT / f'aid1851_nb20.{uuid.uuid4().hex}.tmp'
    import shutil; shutil.copyfile(tmp, tmp_drive); os.replace(tmp_drive, AID)
raw=pd.read_csv(AID, low_memory=False, skiprows=[1])
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
# Max_Response is percent activity at the top concentration, negative for inhibition.
# Negated so larger means more inhibition, and clipped: a handful of rows reach +1330,
# which would set the head's standard deviation on its own.
raw['mr']=(-pd.to_numeric(raw['Max_Response'], errors='coerce')).clip(-50, 150)
raw=raw.dropna(subset=['panel','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS rows', len(raw), '| with fitted AC50', int(raw.pac50.notna().sum()),
      '| with Max_Response', int(raw.mr.notna().sum()))

## Build the target table

Public compounds that are also challenge compounds are dropped, so nothing the blind set
contains leaks in through an auxiliary head. In the `ac50` control this cell reproduces
nb5's 13-column table exactly.

In [ ]:
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
ours={c for c in map(canon, np.concatenate([SM_TR,SM_TE])) if c}

def public_wide(col):
    r=raw.dropna(subset=[col])
    p=r.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES','panel'],as_index=False)[col].median()
    return p.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES',columns='panel',values=col
                         ).reindex(columns=PAN)

blocks = {'ac50': [('qh', 'pac50')], 'mr': [('mr', 'mr')],
          'both': [('qh', 'pac50'), ('mr', 'mr')]}[VARIANT]
wides = [public_wide(col) for _, col in blocks]
idx = wides[0].index
for w in wides[1:]:
    idx = idx.union(w.index)
keep = [s for s in idx if (c := canon(s)) is not None and c not in ours]
wides = [w.reindex(keep) for w in wides]
print('public compounds after dropping overlaps', len(keep))

HEADS=[f'chal_{i}' for i in ISO]+[f'scr_{i}' for i in ISO]
for pre, _ in blocks:
    HEADS += [f'{pre}_{p}' for p in PAN]
n_ch=len(SM_TR); n_pub=len(keep)
SM_ALL=np.concatenate([SM_TR, np.array(keep, dtype=object)])
Y=np.full((n_ch+n_pub, len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j]=AUX[:, j]
for b, w in enumerate(wides):
    Y[n_ch:, 8+5*b:13+5*b]=w.to_numpy(float)
M=np.isfinite(Y)
# Normalisation from development rows only; using all rows would fit a statistic on
# held-out labels. Affine, so it cannot inflate a Spearman, but it should still be clean.
_norm = Y.copy(); _norm[np.flatnonzero(hold)] = np.nan
mu = np.nanmean(_norm, 0); sd = np.nanstd(_norm, 0)
Yz_nan=((Y-mu)/np.where(sd>0,sd,1.0)).astype(np.float32)  # NaN kept as the mask
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, M.sum(0).tolist())))

In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * msk).sum() / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu

## Data check

The whole point is coverage, so stop here if it is not there. In `mr` and `both` the
CYP2D6 `Max_Response` head should cover close to every public compound; in nb5 the
matching head covered about a third.

In [ ]:
for pre, _ in blocks:
    j = HEADS.index(f'{pre}_CYP2D6')
    print(f'{pre}_CYP2D6 labelled on {M[n_ch:, j].sum()} of {n_pub} public compounds')
if VARIANT in ('mr', 'both'):
    j = HEADS.index('mr_CYP2D6')
    assert M[n_ch:, j].mean() > 0.9, 'Max_Response head is not covering the screen'

## Full run

Public compounds sit in training for every fold; only the challenge rows rotate.

In [ ]:
EPOCHS = 25
RESULTS = []
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold); ho_rows = np.flatnonzero(obs_any & hold)
pub_idx = np.arange(n_ch, n_ch + n_pub)

# Checkpoint after every fit, so a disconnect costs one fold rather than the whole run.
# Re-running this cell picks up wherever it stopped.
CKPT = OUT / f'nb20_{VARIANT}_state.npz'
if CKPT.exists():
    st = np.load(CKPT, allow_pickle=True)
    oof_all = st['oof_all']; done = set(st['done'].tolist())
    hp = st['hp'] if st['hp'].size else None
    tp = st['tp'] if st['tp'].size else None
    print('resuming; folds already done:', sorted(done), '| holdout',
          hp is not None, '| test', tp is not None)
else:
    oof_all = np.full((n_ch, 4), np.nan, np.float32); done = set(); hp = tp = None

def save():
    np.savez_compressed(CKPT, oof_all=oof_all, done=np.array(sorted(done)),
                        hp=hp if hp is not None else np.array([]),
                        tp=tp if tp is not None else np.array([]))

t0 = time.time()
for f, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    if f in done:
        continue
    m = train(np.concatenate([dev_rows[tr], pub_idx]), epochs=EPOCHS)
    oof_all[dev_rows[va]] = predict(m, dev_rows[va])[:, :4]
    done.add(f); save()
    print(f'  fold {f} done, checkpointed ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if hp is None:
    m = train(np.concatenate([dev_rows, pub_idx]), epochs=EPOCHS)
    hp = predict(m, ho_rows)[:, :4]; save()
    print(f'  holdout model done ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if tp is None:
    mt = train(np.concatenate([dev_rows, ho_rows, pub_idx]), epochs=EPOCHS)
    tp = predict(mt, test=True)[:, :4]; save()
    print(f'  test model done ({time.time()-t0:.0f}s)', flush=True)
    del mt; gc.collect(); torch.cuda.empty_cache()

for j, i in enumerate(ISO):
    y = y_all[i]
    d = dev_rows[np.isfinite(y[dev_rows])]; h = ho_rows[np.isfinite(y[ho_rows])]
    np.savez_compressed(OUT / f'{ARM}_{i}.npz', oof=oof_all[d, j], dev=d,
                        hold=hp[np.isin(ho_rows, h), j], ho=h, test=tp[:, j])
    r = float(spearmanr(y[d], oof_all[d, j]).statistic)
    rh = float(spearmanr(y[h], hp[np.isin(ho_rows, h), j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh))
    json.dump(RESULTS, open(OUT / f'results_nb20_{VARIANT}.json', 'w'), indent=2)
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)
print('\nmtl13 (nb5), same architecture and qHTS AC50 heads: 0.5577 0.6935 0.4515 0.8022')
print('  macro dev 0.6262 | macro holdout 0.6652')
print('%s macro dev %.4f | macro holdout %.4f' % (ARM,
    np.mean([x['dev_spearman'] for x in RESULTS]),
    np.mean([x['holdout_spearman'] for x in RESULTS])))

# Only this run's own files: zipping the whole folder would read nb19's checkpoint while
# nb19 may be writing it.
import zipfile
mine = [CKPT, OUT / f'results_nb20_{VARIANT}.json'] + [OUT / f'{ARM}_{i}.npz' for i in ISO]
with zipfile.ZipFile(f'/content/nb20_{VARIANT}_results.zip', 'w') as z:
    for p in mine:
        z.write(p, p.name)
try:
    from google.colab import files
    files.download(f'/content/nb20_{VARIANT}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- results are on Drive in', OUT)